# Recurrent neural networks

In this tutorial, we'll build a Recurrent Neural Network (RNN) to classify DNA sequences using PyTorch and PyTorch Lightning. This is a common bioinformatics task that's well-suited for sequence models.

## Introduction to RNNs

Recurrent Neural Networks (RNNs) are a class of neural networks designed to work with sequential data. Unlike feed-forward networks, RNNs have connections that form cycles, allowing information to persist across processing steps. This makes them particularly useful for tasks like:

- Natural language processing
- Time series prediction
- **Biological sequence analysis (our focus today)**

RNNs process sequences one element at a time while maintaining an internal "memory" of what they've seen before. This is perfect for DNA sequences, which are naturally sequential data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L

from torch.utils.data import DataLoader, Dataset, Subset, random_split
from lightning.pytorch.loggers import CSVLogger
from sklearn.metrics import classification_report

torch.set_float32_matmul_precision("high")

## Exploring the splice site prediction data set

The DNA sequences we'll be using today are each representing 

In [ ]:
TRAIN_DATA_PATH = "https://raw.githubusercontent.com/CompOmics/D012554A_2025/refs/heads/main/data/splicing/acceptor_sites_dataset_train.csv"
TEST_DATA_PATH = "https://raw.githubusercontent.com/CompOmics/D012554A_2025/refs/heads/main/data/splicing/acceptor_sites_dataset_test.csv"

In [ ]:
train_data_df = pd.read_csv(TRAIN_DATA_PATH).head()
train_data_df.head()

In [ ]:
train_data_df["sequence"].str.len().describe()

All sequences are of length 22. This makes sense, as they have all been selected from large genomics data sets, exactly 22 base pair positions around the (putative) splice site. Negative examples are also sequences of length 22, with the obligatory `AG` sequence in the middle.

Question: How does the consistent length simplify dealing with this data in the context of machine learning?

In [ ]:
# Add your answer here


Let's further explore the data. We can simply plot the nucleotide counts at each location along the sequence and check if we can already detect any patterns:

In [ ]:
position_counts = (
    pd.read_csv(TRAIN_DATA_PATH)
    .assign(target=lambda d: (d["label"] != -1).astype(int))
    .set_index("target")["sequence"]
    .str.extractall(r"(.)")
    .rename(columns={0: "nucleotide"})
    .reset_index()
    .rename(columns={"level_1": "sample", "match": "position"})
    .groupby(["target", "position", "nucleotide"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

g = sns.catplot(
    data=position_counts,
    kind="bar",
    x="position",
    y="count",
    hue="nucleotide",
    row="target",
    row_order=[0, 1],
    estimator="sum",
    errorbar=None,
    height=3,
    aspect=4,
    sharey=False,
)
g.set_titles("target = {row_name}")
g.set_axis_labels("Position", "Count")
plt.show()

Question: What do you observe?

In [ ]:
# Add your answer here

## Preparing the data classes

### Encoding the nucleotides as integer tokens

In [ ]:
NUCLEOTIDE_TO_IDX = {"A": 1, "C": 2, "G": 3, "T": 4}
VOCAB_SIZE = len(NUCLEOTIDE_TO_IDX) + 1  # reserve index 0

def encode_sequence(sequence: str) -> torch.Tensor:
    return torch.tensor([NUCLEOTIDE_TO_IDX[nucleotide] for nucleotide in sequence], dtype=torch.long)

Task: test the encode_sequence function with some random nucleotide sequences. What does it do?

In [ ]:
# Write your code here



### PyTorch Dataset class

Task: Complete the dataset class `__getitem__` method:

In [ ]:
class DNASequenceDataset(Dataset):
    def __init__(self, sequences, labels=None):
        self.sequences = list(sequences)

        if labels is None:
            labels = [float("nan")] * len(self.sequences)
        self.labels = torch.tensor(list(labels), dtype=torch.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        label = self.labels[idx]

        ### Write your code here
        encoded_sequence = encode_sequence(self.sequences[idx])
        ###
        return encoded_sequence, label

### Create datasets and data loaders

We'll let the `LightningDataModule` handle loading, preprocessing, and splitting the dataset.

In [ ]:
class DNASequenceDataModule(L.LightningDataModule):
    def __init__(
        self,
        train_data_path: str,
        test_data_path: str,
        batch_size: int = 64,
        val_frac: float = 0.15,
        seed: int = 42,
    ):
        super().__init__()
        self.train_data_path = train_data_path
        self.test_data_path = test_data_path
        self.batch_size = batch_size
        self.val_frac = val_frac
        self.seed = seed

        self.train_df = None
        self.test_df = None

    def setup(self, stage=None):
        # Load train data
        self.train_df = pd.read_csv(self.train_data_path).copy()
        self.train_df["label"] = (self.train_df["label"] != -1).astype(int)

        # Load test data
        self.test_df = pd.read_csv(self.test_data_path).copy()
        if "label" in self.test_df.columns:
            self.test_df["label"] = (self.test_df["label"] != -1).astype(int)

        # Create dataset objects
        full_train_dataset = DNASequenceDataset(self.train_df["sequence"], self.train_df["label"])

        # Split train dataset into train and validation sets
        n_total = len(full_train_dataset)
        n_val = int(n_total * self.val_frac)
        n_train = n_total - n_val

        self.train_ds, self.val_ds = random_split(
            full_train_dataset,
            [n_train, n_val],
            generator=torch.Generator().manual_seed(self.seed),
        )

        self.test_ds = DNASequenceDataset(
            self.test_df["sequence"],
            self.test_df["label"] if "label" in self.test_df.columns else None,
        )

    def _dataloader(self, dataset, shuffle: bool = False):
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=shuffle,
            num_workers=0,
        )

    def train_dataloader(self):
        return self._dataloader(self.train_ds, shuffle=True)

    def val_dataloader(self):
        return self._dataloader(self.val_ds)

    def test_dataloader(self):
        return self._dataloader(self.test_ds)

## Building the RNN Model

Now, let's define our RNN model for DNA sequence classification.

Our model will consist of the following layers:

- Embedding
- GRU
- Dropout
- Linear
- Loss

The dropout layer is a form of regularization, the loss function calculates the loss metric. 

Question: What functions do the other layers perform?

In [ ]:
# Write your answer here


In [ ]:
class DNASequenceGRUClassifier(L.LightningModule):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_dim: int,
        num_layers: int = 1,
        dropout: float = 0.2,
        lr: float = 1e-3,
    ):
        super().__init__()
        self.save_hyperparameters()  # also sets learning rate as self.hparams.lr

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, 1)
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, sequences):
        embedded = self.embedding(sequences)
        out, _ = self.rnn(embedded)
        features = self.dropout(out[:, -1, :])
        return self.classifier(features).squeeze(-1)

    def _shared_step(self, batch, stage: str):
        sequences, labels = batch
        labels = labels.float()
        logits = self(sequences)
        loss = self.loss_fn(logits, labels)
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=sequences.size(0))
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def predict_step(self, batch, batch_idx):
        sequences, labels = batch
        probs = torch.sigmoid(self.forward(sequences))
        return {"probs": probs, "labels": labels}

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

### Initialize the data classes

In [ ]:
data_module = DNASequenceDataModule(
    train_data_path=TRAIN_DATA_PATH,
    test_data_path=TEST_DATA_PATH,
    batch_size=64,
)
data_module.setup()

Let's print some statistics on the datasets:

In [ ]:
print(f"Train samples: {len(data_module.train_df)}")
print(f"Test samples: {len(data_module.test_df)}")
print(data_module.train_df["label"].value_counts().sort_index().rename(index={0: "negative", 1: "positive"}))

### Initialize the model

In [ ]:
model = DNASequenceGRUClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=8,
    hidden_dim=64,
    num_layers=2,
    dropout=0.2,
    lr=1e-3,
)
model

## Training the Model



In [ ]:
logger = CSVLogger(save_dir=".", name="lightning_logs", version="dna_gru")
trainer = L.Trainer(
    max_epochs=20,
    accelerator="auto",
    logger=logger,
    log_every_n_steps=1,
)

trainer.fit(model, datamodule=data_module)

## Evaluating and Testing

### Evaluating the learning curves

In [ ]:
metrics_path = Path(trainer.logger.log_dir) / "metrics.csv"
metrics = pd.read_csv(metrics_path)

learning_curves = (
    metrics.groupby("epoch", as_index=False)[["train_loss", "val_loss"]]
    .last()
    .sort_values("epoch")
)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
learning_curves.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax, marker="o", title="Training and validation loss")
ax.set_ylabel("Binary cross-entropy loss")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Evaluating the final model on the test set

In [ ]:
test_loader = data_module.test_dataloader()
test_metrics = trainer.test(model, dataloaders=test_loader, verbose=False)[0]
test_outputs = trainer.predict(model, dataloaders=test_loader)

test_probabilities = torch.cat([output["probs"] for output in test_outputs]).cpu().numpy().ravel()
test_targets = torch.cat([output["labels"] for output in test_outputs]).cpu().numpy().astype(int)

test_predictions = (test_probabilities >= 0.5).astype(int)

print(pd.DataFrame([test_metrics]).round(4))
print(classification_report(test_targets, test_predictions, digits=4))

### Making predictions with the trained model

In [ ]:
example_sequence = "ATCGATCGATCGATCGATCGAT"

example_dataset = DNASequenceDataset([example_sequence])
example_loader = DataLoader(example_dataset, batch_size=1)

example_output = trainer.predict(model, dataloaders=example_loader)[0]
example_probability = float(example_output["probs"].cpu().item())

predicted_class = int(example_probability >= 0.5)

print(f"Sequence: {example_sequence}")
print(f"Class 1 probability: {example_probability:.4f}")
print(f"Predicted class: {predicted_class}")

Question: What is the model saying? Is our peptide likely a splice site acceptor or not?

In [ ]:
# Write your answer here


## Exercise: Hyperparameter sweep with WandB

As in the first PyTorch Lightning practical, define a Weights and Biases Sweep to find the optimal
values for these hyperparameters:

- embedding_dim
- hidden_dim
- num_layers

In [ ]:
import wandb

Setup the function that defines testing a single run within the sweep:

In [ ]:
def sweep_run(*args, **kwargs):
    trainer =   # Write your code here

    trainer.fit(
        ### Write your code here



        ###
    )

    wandb.finish()

Setup the sweep configuration, including the hyperparameter ranges:

In [ ]:
sweep_configuration = {
    "method": "random",
    "metric": {"goal": "minimize", "name": "val_loss"},
    "parameters": {
        ### Write your code here


        ###
    },
}

Initialize the sweep:

In [ ]:
sweep_id = wandb.sweep(sweep=sweep_configuration, project="splice-site-classification")

Start the sweep!

In [ ]:
wandb.agent(sweep_id, function=sweep_run, count=20)

Question: What are the results? What hyperparameters work best? Is this as you expected?

In [ ]:
# Write your answer here
